# 04 Export + Quantization + Latency
Notebook-first deployment benchmarking for ShiroNet edge model.

## Exports
- TorchScript
- ONNX
- Dynamic quantized state dict

## Latency table
- FP32 CPU
- TorchScript CPU
- Dynamic INT8 CPU
- Optional GPU if available

In [ ]:
import json, time, torch
from pathlib import Path
from src.models.factory import create_model

In [ ]:
ckpt_path = Path('models/intel_shironet_edge/shironet_shufflenet_v2_x0_5.pt')
out_dir = Path('docs/assets/optimization/export_benchmark')
out_dir.mkdir(parents=True, exist_ok=True)
ckpt = torch.load(ckpt_path, map_location='cpu')
arch = ckpt.get('arch', 'shufflenet_v2_x0_5')
classes = ckpt['classes']
img_size = int(ckpt.get('img_size', 160))
model = create_model(arch, num_classes=len(classes), pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('arch:', arch, 'classes:', len(classes), 'img_size:', img_size)

In [ ]:
def bench(m, device='cpu', bs=1, warm=10, steps=100):
    m = m.to(device).eval()
    x = torch.randn(bs, 3, img_size, img_size, device=device)
    with torch.no_grad():
        for _ in range(warm):
            _ = m(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(steps):
            _ = m(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    return (t1 - t0) * 1000 / steps

lat = {}
lat['fp32_cpu'] = bench(model, 'cpu')
if torch.cuda.is_available():
    g = create_model(arch, len(classes), False)
    g.load_state_dict(ckpt['model_state_dict'])
    lat['fp32_gpu'] = bench(g, 'cuda')
else:
    lat['fp32_gpu'] = None

example = torch.randn(1,3,img_size,img_size)
ts = torch.jit.trace(model.cpu(), example)
ts.save(str(out_dir / f'{arch}_ts.pt'))
lat['torchscript_cpu'] = bench(ts, 'cpu')

q = torch.quantization.quantize_dynamic(model.cpu(), {torch.nn.Linear}, dtype=torch.qint8)
torch.save(q.state_dict(), out_dir / f'{arch}_dynamic_q.pt')
lat['dynamic_int8_cpu'] = bench(q, 'cpu')

print(lat)

In [ ]:
onnx_path = out_dir / f'{arch}.onnx'
onnx_ok, onnx_err = True, ''
try:
    torch.onnx.export(model.cpu(), torch.randn(1,3,img_size,img_size), str(onnx_path), input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0:'batch'}, 'logits': {0:'batch'}}, opset_version=17)
except Exception as e:
    onnx_ok, onnx_err = False, str(e)

payload = {
    'arch': arch,
    'checkpoint': str(ckpt_path),
    'img_size': img_size,
    'latency_ms': lat,
    'onnx_export': {'success': onnx_ok, 'error': onnx_err},
}
(out_dir / 'export_benchmark.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
print(json.dumps(payload, indent=2))

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'Variant':'FP32 CPU', 'Latency(ms)': payload['latency_ms']['fp32_cpu']},
    {'Variant':'FP32 GPU', 'Latency(ms)': payload['latency_ms']['fp32_gpu']},
    {'Variant':'TorchScript CPU', 'Latency(ms)': payload['latency_ms']['torchscript_cpu']},
    {'Variant':'Dynamic INT8 CPU', 'Latency(ms)': payload['latency_ms']['dynamic_int8_cpu']},
])
display(df)